### Metrics for all-RFH Patching

In [1]:
import pandas as pd
import os
from pathlib import Path
model_alias = "qwen-7B"
path = f"data/output/activation_patching/{model_alias}/all_rfh_heads.jsonl"
df = pd.read_json(path, lines=True)
df.head()

,withR_loss,withR_joint_logprob,withR_mean_logprob,withoutR_loss,withoutR_joint_logprob,withoutR_mean_logprob,patched_withoutR_loss,patched_withoutR_joint_logprob,patched_withoutR_mean_logprob,patched_withR_loss,patched_withR_joint_logprob,patched_withR_mean_logprob
0,0.462891,-6.93750,-0.462891,0.558594,-8.37500,-0.558594,0.546875,-8.187500,-0.546875,0.455078,-6.84375,-0.455078
1,0.890625,-7.12500,-0.890625,1.648438,-13.18750,-1.648438,1.281250,-10.250000,-1.281250,0.914062,-7.31250,-0.914062
2,0.628906,-7.56250,-0.628906,1.007812,-12.12500,-1.007812,0.898438,-10.812500,-0.898438,0.617188,-7.40625,-0.617188
3,1.031250,-6.18750,-1.031250,0.660156,-3.96875,-0.660156,0.578125,-3.453125,-0.578125,1.015625,-6.12500,-1.015625
4,0.863281,-6.90625,-0.863281,1.000000,-8.00000,-1.000000,0.960938,-7.687500,-0.960938,0.855469,-6.84375,-0.855469


In [5]:
import pandas as pd
import plotly.graph_objects as go

def plot_medidation_metrics(df: pd.DataFrame, show: bool = True):

    # Map to A/B/C/D notation (means over dataset)
    A = df["withR_loss"].mean()              # R=1, RFH=1
    B = df["withoutR_loss"].mean()           # R=0, RFH=0
    C = df["patched_withR_loss"].mean()      # R=1, RFH=0
    D = df["patched_withoutR_loss"].mean()   # R=0, RFH=1

    # Effects on loss scale (positive = reduction in loss, i.e., better)
    te   = B - A   # Total Effect of CoT
    nde  = B - C   # Pure Natural Direct Effect (not via RFHs)
    nie  = B - D   # Pure Natural Indirect Effect (via RFHs, R=0 world)
    tnie = C - A   # Total Natural Indirect Effect in CoT world

    # ---------- TE PLOT (B vs A) ----------
    x_te = [
        "Baseline<br>R=0, RFH=0 (B)",
        "CoT + RFH<br>R=1, RFH=1 (A)",
    ]
    y_te = [B, A]

    fig_te = go.Figure()
    fig_te.add_trace(
        go.Bar(
            x=x_te,
            y=y_te,
            text=[f"{v:.3f}" for v in y_te],
            textposition="outside",
            name="Cross-entropy loss",
        )
    )
    fig_te.add_annotation(
        x=0.5,
        y=max(y_te) * 1.05,
        xref="paper",
        yref="y",
        text=f"TE = B - A = {te:.3f}",
        showarrow=False,
        font=dict(size=14),
    )
    fig_te.update_layout(
        title=dict(
            text="Total Effect (TE) of CoT on Loss",
            x=0.5,
            xanchor="center",
            font=dict(size=20),
        ),
        xaxis=dict(title="Condition"),
        yaxis=dict(title="Cross-entropy loss"),
        template="plotly_white",
        font=dict(family="Helvetica, Arial, sans-serif", size=14),
        margin=dict(l=70, r=40, t=80, b=90),
    )

    # ---------- NDE PLOT (B vs C) ----------
    x_nde = [
        "Baseline<br>R=0, RFH=0 (B)",
        "CoT, RFH=0<br>R=1, RFH=0 (C)",
    ]
    y_nde = [B, C]

    fig_nde = go.Figure()
    fig_nde.add_trace(
        go.Bar(
            x=x_nde,
            y=y_nde,
            text=[f"{v:.3f}" for v in y_nde],
            textposition="outside",
            name="Cross-entropy loss",
        )
    )
    fig_nde.add_annotation(
        x=0.5,
        y=max(y_nde) * 1.05,
        xref="paper",
        yref="y",
        text=f"NDE = B - C = {nde:.3f}",
        showarrow=False,
        font=dict(size=14),
    )
    fig_nde.update_layout(
        title=dict(
            text="Pure Natural Direct Effect (NDE) on Loss",
            x=0.5,
            xanchor="center",
            font=dict(size=20),
        ),
        xaxis=dict(title="Condition"),
        yaxis=dict(title="Cross-entropy loss"),
        template="plotly_white",
        font=dict(family="Helvetica, Arial, sans-serif", size=14),
        margin=dict(l=70, r=40, t=80, b=90),
    )

    # ---------- NIE PLOT (B vs D) ----------
    x_nie = [
        "Baseline<br>R=0, RFH=0 (B)",
        "RFH-only<br>R=0, RFH=1 (D)",
    ]
    y_nie = [B, D]

    fig_nie = go.Figure()
    fig_nie.add_trace(
        go.Bar(
            x=x_nie,
            y=y_nie,
            text=[f"{v:.3f}" for v in y_nie],
            textposition="outside",
            name="Cross-entropy loss",
        )
    )
    fig_nie.add_annotation(
        x=0.5,
        y=max(y_nie) * 1.05,
        xref="paper",
        yref="y",
        text=f"NIE = B - D = {nie:.3f}",
        showarrow=False,
        font=dict(size=14),
    )
    fig_nie.update_layout(
        title=dict(
            text="Pure Natural Indirect Effect (NIE) via RFHs on Loss",
            x=0.5,
            xanchor="center",
            font=dict(size=20),
        ),
        xaxis=dict(title="Condition"),
        yaxis=dict(title="Cross-entropy loss"),
        template="plotly_white",
        font=dict(family="Helvetica, Arial, sans-serif", size=14),
        margin=dict(l=70, r=40, t=80, b=90),
    )

    # ---------- TNIE PLOT (C vs A) ----------
    x_tnie = [
        "CoT, RFH=0<br>R=1, RFH=0 (C)",
        "CoT + RFH<br>R=1, RFH=1 (A)",
    ]
    y_tnie = [C, A]

    fig_tnie = go.Figure()
    fig_tnie.add_trace(
        go.Bar(
            x=x_tnie,
            y=y_tnie,
            text=[f"{v:.3f}" for v in y_tnie],
            textposition="outside",
            name="Cross-entropy loss",
        )
    )
    fig_tnie.add_annotation(
        x=0.5,
        y=max(y_tnie) * 1.05,
        xref="paper",
        yref="y",
        text=f"TNIE = C - A = {tnie:.3f}",
        showarrow=False,
        font=dict(size=14),
    )
    fig_tnie.update_layout(
        title=dict(
            text="Total Natural Indirect Effect (TNIE) in CoT World",
            x=0.5,
            xanchor="center",
            font=dict(size=20),
        ),
        xaxis=dict(title="Condition"),
        yaxis=dict(title="Cross-entropy loss"),
        template="plotly_white",
        font=dict(family="Helvetica, Arial, sans-serif", size=14),
        margin=dict(l=70, r=40, t=80, b=90),
    )

    if show:
        fig_te.show()
        fig_nde.show()
        fig_nie.show()
        fig_tnie.show()
    else:
        output_dir = f"data/graphs"
        os.makedirs(Path(output_dir), exist_ok=True)
        # You'll need model_alias defined in your scope if you keep these filenames
        fig_te.write_image(f"{output_dir}/te_bar_{model_alias}.png", scale=2)
        fig_nde.write_image(f"{output_dir}/nde_bar_{model_alias}.png", scale=2)
        fig_nie.write_image(f"{output_dir}/nie_bar_{model_alias}.png", scale=2)
        fig_tnie.write_image(f"{output_dir}/tnie_bar_{model_alias}.png", scale=2)

    return te.item(), nde.item(), nie.item(), tnie.item()

te, nde, nie, tnie = plot_medidation_metrics(df, show=False)
print("Total Effect (TE):", te)
print("Natural Direct Effect (NDE):", nde)
print("Natural Indirect Effect (NIE):", nie)
print("Total Natural Indirect Effect (TNIE):", tnie)
print("NIE as % of TE:", (nie / te) * 100)

Total Effect (TE): 0.15513245488556338
Natural Direct Effect (NDE): 0.162841796875
Natural Indirect Effect (NIE): 0.090882207306338
Total Natural Indirect Effect (TNIE): -0.007709341989436624
NIE as % of TE: 58.58361963870108


### Helper Functions

In [2]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go

LOSS_COLS = [
    "withR_loss",
    "withoutR_loss",
    "patched_withoutR_loss",
    "patched_withR_loss",
]

def group_mean_losses(df: pd.DataFrame, group_key: str, sort_key: str | None = None):
    """Group by group_key, average LOSS_COLS, and sort."""
    grouped = (
        df.groupby(group_key, as_index=False)[LOSS_COLS]
          .mean()
    )
    if sort_key is None:
        sort_key = group_key
    grouped = grouped.sort_values(sort_key).reset_index(drop=True)
    return grouped

def add_mediation_columns(grouped: pd.DataFrame):
    """Add TE/NDE/NIE/TNIE and NIE_over_TE_pct columns in-place + return view."""
    A = grouped["withR_loss"]
    B = grouped["withoutR_loss"]
    C = grouped["patched_withR_loss"]
    D = grouped["patched_withoutR_loss"]

    grouped["TE"]   = B - A
    grouped["NDE"]  = B - C
    grouped["NIE"]  = B - D
    grouped["TNIE"] = C - A

    grouped["NIE_over_TE_pct"] = np.where(
        np.isclose(grouped["TE"], 0.0),
        np.nan,
        (grouped["NIE"] / grouped["TE"]) * 100.0
    )
    return grouped

def save_or_show(fig, show: bool, output_dir: str, filename: str, model_alias: str | None):
    """Unified show/save logic."""
    if show:
        fig.show()
        return
    if model_alias is None:
        model_alias = "model"
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    fig.write_image(f"{output_dir}/{filename}_{model_alias}.png", scale=2)

def bar_pct_figure(x, y, title, x_title, y_title, hover_label):
    """Standard bar chart for percentage plots."""
    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=x,
            y=y,
            text=[f"{v:.1f}%" if pd.notna(v) else "NaN" for v in y],
            textposition="outside",
            name="NIE / TE (%)",
            hovertemplate=f"{hover_label} %{{x}}<br>NIE/TE: %{{y:.2f}}%<extra></extra>",
        )
    )
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center", font=dict(size=14)),
        xaxis=dict(title=x_title, tickmode="linear"),
        yaxis=dict(title=y_title),
        template="plotly_white",
        font=dict(family="Helvetica, Arial, sans-serif", size=14),
        margin=dict(l=70, r=40, t=80, b=70),
    )
    return fig


### Metrics for Layer-wise Patching

In [11]:
import pandas as pd

layer_map = {
    "qwen-1p5B": [1, 12, 14, 16, 19, 20, 23],
    "qwen-7B": [16, 19, 14, 1, 17, 22]
}
dirpath = f"data/output/activation_patching/{model_alias}"

layer_df = []
for l in layer_map[model_alias]:
    path = f"{dirpath}/layer_{l}_rfh_heads.jsonl"
    temp = pd.read_json(path, lines=True)
    # add the layer number as a column
    temp['layer'] = l
    layer_df.append(temp)
# concatenate all dataframes
layer_df = pd.concat(layer_df, ignore_index=True)

In [12]:
def plot_layerwise_metrics(df, show=True, model_alias=None, output_dir="data/graphs"):
    grouped = group_mean_losses(df, group_key="layer")
    grouped = add_mediation_columns(grouped)

    layers = grouped["layer"].tolist()
    ratios = grouped["NIE_over_TE_pct"].tolist()

    fig = bar_pct_figure(
        x=layers,
        y=ratios,
        title="Layer-wise % of CoT Performance Recovered via RFH Patching (NIE/TE)",
        x_title="Layer",
        y_title="NIE / TE (%)",
        hover_label="Layer"
    )

    save_or_show(fig, show, output_dir, "layerwise_nie_over_te", model_alias)

    return fig, grouped[["layer","TE","NDE","NIE","TNIE","NIE_over_TE_pct"]]

figs, res = plot_layerwise_metrics(layer_df, show = False, model_alias = model_alias)
res.head(10)

,layer,TE,NDE,NIE,TNIE,NIE_over_TE_pct
0,1,0.155132,0.157495,0.001413,-0.002362,0.911005
1,14,0.155132,0.160438,0.005474,-0.005306,3.528760
2,16,0.155132,0.152983,0.021278,0.002149,13.716059
3,17,0.155132,0.145374,0.009222,0.009759,5.944808
4,19,0.155132,0.160108,0.020202,-0.004976,13.022276
5,22,0.155132,0.163045,0.037494,-0.007912,24.169345


### Metrics for Head-wise Patching

In [13]:
import pandas as pd
from src.constants import TOP_RFHS_BY_LAYER_HEAD

layer_to_top_rfhs = {}
for k, v in TOP_RFHS_BY_LAYER_HEAD.items():
    d_tmp = {}
    for cord in v:
        if cord[0] not in d_tmp:
            d_tmp[cord[0]] = []
        d_tmp[cord[0]].append(cord[1])
    layer_to_top_rfhs[k] = d_tmp
layer_to_top_rfhs = layer_to_top_rfhs[model_alias]

dirpath = f"data/output/activation_patching/{model_alias}"

layer_head_df = []
for l in layer_to_top_rfhs:
    for head in layer_to_top_rfhs[l]:    
        path = f"{dirpath}/layer_{l}_rfh_head_{head}.jsonl"
        temp = pd.read_json(path, lines=True)
        # add the layer number as a column
        temp['layer_head'] = f"{l}-{head}"
        layer_head_df.append(temp)
# concatenate all dataframes
layer_head_df = pd.concat(layer_head_df, ignore_index=True)

In [14]:
def plot_headlayerwise_metrics(df, show=True, model_alias=None, output_dir="data/graphs"):
    grouped = group_mean_losses(df, group_key="layer_head")
    grouped = add_mediation_columns(grouped)

    grouped = grouped.sort_values(
        by="NIE_over_TE_pct", ascending=False, na_position="last"
    ).reset_index(drop=True)

    x = grouped["layer_head"].tolist()
    y = grouped["NIE_over_TE_pct"].tolist()

    fig = bar_pct_figure(
        x=x,
        y=y,
        title="Layer-Head-wise % of CoT Performance Recovered via RFH Patching (NIE/TE)",
        x_title="Layer-Head",
        y_title="NIE / TE (%)",
        hover_label="Layer-Head"
    )
    fig.update_layout(xaxis=dict(title="Layer-Head", type="category"))

    save_or_show(fig, show, output_dir, "layerhead_wise_nie_over_te", model_alias)

    return fig, grouped[["layer_head","TE","NDE","NIE","TNIE","NIE_over_TE_pct"]]

figs, res = plot_headlayerwise_metrics(layer_head_df, show=False, model_alias=model_alias)
res.head(20)


,layer_head,TE,NDE,NIE,TNIE,NIE_over_TE_pct
0,22-7,0.155132,0.163045,0.037494,-0.007912,24.169345
1,19-15,0.155132,0.160108,0.020202,-0.004976,13.022276
2,16-0,0.155132,0.154434,0.014841,0.000698,9.566663
3,17-18,0.155132,0.145322,0.013538,0.009810,8.726588
4,14-0,0.155132,0.164678,0.008476,-0.009546,5.463815
5,16-14,0.155132,0.154737,0.008335,0.000395,5.372936
6,1-1,0.155132,0.157495,0.001413,-0.002362,0.911005
7,17-14,0.155132,0.150356,0.000653,0.004776,0.421146
8,14-7,0.155132,0.151756,-0.004236,0.003377,-2.730799
9,17-19,0.155132,0.160301,-0.006207,-0.005168,-4.000887


### Metrics for Top-k Patching

In [ ]:
import pandas as pd
from src.constants import TOP_RFHS_BY_LAYER_HEAD

layer_to_top_rfhs = {}
for k, v in TOP_RFHS_BY_LAYER_HEAD.items():
    d_tmp = {}
    for cord in v:
        if cord[0] not in d_tmp:
            d_tmp[cord[0]] = []
        d_tmp[cord[0]].append(cord[1])
    layer_to_top_rfhs[k] = d_tmp
layer_to_top_rfhs = layer_to_top_rfhs[model_alias]

dirpath = f"data/output/activation_patching/{model_alias}"
topk_df = []
for k in range(1, 6):
    path = f"{dirpath}/top_{k}_rfh.jsonl"
    temp = pd.read_json(path, lines=True)
    # add the layer number as a column
    temp['topk'] = k
    topk_df.append(temp)
# concatenate all dataframes
topk_df = pd.concat(topk_df, ignore_index=True)

In [8]:
def plot_topk_metrics(df, show=True, model_alias=None, output_dir="data/graphs"):
    grouped = group_mean_losses(df, group_key="topk")
    grouped = add_mediation_columns(grouped)

    grouped = grouped.sort_values(
        by="NIE_over_TE_pct", ascending=False, na_position="last"
    ).reset_index(drop=True)

    x = grouped["topk"].tolist()
    y = grouped["NIE_over_TE_pct"].tolist()

    fig = bar_pct_figure(
        x=x,
        y=y,
        title="Top-K % of CoT Performance Recovered via RFH Patching (NIE/TE)",
        x_title="Top K",
        y_title="NIE / TE (%)",
        hover_label="Top K"
    )
    fig.update_layout(xaxis=dict(title="Top K", type="category"))
    save_or_show(fig, show, output_dir, "topk_nie_over_te", model_alias)

    return fig, grouped[["topk","TE","NDE","NIE","TNIE","NIE_over_TE_pct"]]

figs, res = plot_topk_metrics(topk_df, show=False, model_alias=model_alias)
res.head(20)

,topk,TE,NDE,NIE,TNIE,NIE_over_TE_pct
0,5,0.546559,0.533507,0.291851,0.013052,53.397816
1,4,0.546559,0.533974,0.277689,0.012585,50.806719
2,3,0.546559,0.538716,0.226276,0.007844,41.400102
3,2,0.546559,0.544565,0.186292,0.001995,34.084545
4,1,0.546559,0.548395,0.120828,-0.001836,22.107034
